# Satellite Data Analysis for Jamaica
## Notebook 3 of 4. Measure Hurricane Melissa's Damage

**Prepared by Adrian Dunkley, Climate Studies Group Mona, Faculty of Science and Technology, University of the West Indies.**

> **STUDENT EDITION.** Cells marked **YOUR TURN** have gaps to fill in. Look
> for `____` and `# TODO`. Many gaps list three options in a comment; one is
> right and the others teach you something by being wrong. Everything else runs
> as given. If you get stuck, read the hint under the cell before asking.
> The day: two hours of missions, then the one-hour Satellite Challenge.

---

# Mission 3: Measure Hurricane Melissa's Damage in Hectares 🌀

**Your question.** How much did Hurricane Melissa destroy, and can you prove it
was the storm and not just the change of season?

**Why it matters.** Relief goes where the numbers point. A red map convinces
nobody. Hectares of proven damage move money, water and people.

**Your objective.** Measure the severe vegetation loss around New Hope,
Westmoreland, where Melissa's eye came ashore. First to the number wins.

On 28 October 2025, Melissa struck at Category 5 with 185 mph (295 km/h) winds,
the strongest storm ever recorded to hit Jamaica. Black River took the worst of
it. Today you measure what it did to the land.

### What you will be able to do by the end

1. Build a matched before picture and after picture around a dated event
2. Subtract the two and read the difference map honestly
3. Sort damage into severity classes and report each in hectares
4. Run a control that separates the storm from the season

**Time in class:** about 30 minutes. The two long cells run for two to three
minutes each; start them, then keep listening.
**Before you start:** Notebooks 1 and 2.

### Words for this notebook

| Word | What it means here |
|---|---|
| NDVI | the Normalized Difference Vegetation Index, the plant-health number from Notebook 2 |
| baseline | The "before" that change is measured against |
| control | The same test run over a period when nothing happened |
| noise floor | The change your method reports when nothing happened |
| severity class | A named band of damage, counted in hectares |
| false positive | A change flagged by the method that is not real on the ground |
| shoreline | The line where the data says land meets water |

---

## Part 1. Setting up

In [ ]:
# Run this once. On Google Colab it takes about a minute.
# If a package is already there, pip will say so and move on.
!pip install -q rasterio requests imageio pandas scikit-learn matplotlib pillow

print("Packages ready.")

In [ ]:
# 🚚 JUST RUN THIS CELL. Nothing to change. It is the toolbox for the whole course.
# ============================================================================
#  JAMAICA EARTH OBSERVATION TOOLKIT
#  Run this cell in every session. It sets up the connection to the satellite
#  archive and defines the handful of functions the whole course uses.
# ============================================================================
import os, math, json, time, warnings
warnings.filterwarnings("ignore")

# GDAL reads the satellite files straight off Amazon's servers over the
# internet. These settings tell it how to behave: no login needed, do not list
# whole directories, retry if the network hiccups.
os.environ.update({
    "AWS_NO_SIGN_REQUEST": "YES",
    "GDAL_HTTP_UNSAFESSL": "YES",
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
    "GDAL_HTTP_MAX_RETRY": "5",
    "GDAL_HTTP_RETRY_DELAY": "2",
})

import requests, numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
from rasterio.warp import Resampling
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.vrt import WarpedVRT
from PIL import Image, ImageDraw

STAC_URL = "https://earth-search.aws.element84.com/v1/search"

def _stac_post(url, body, timeout=60, tries=4):
    """POST to the archive, retrying politely if the server is having a moment."""
    for attempt in range(tries):
        try:
            r = requests.post(url, json=body, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException:
            if attempt == tries - 1:
                raise
            time.sleep(2 * (attempt + 1))    # 2 s, 4 s, 6 s between tries


# House style for every chart in this course.
CYAN, INK, SAND = "#00b8d4", "#12232e", "#e0a458"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 13,
    "axes.titleweight": "bold", "figure.facecolor": "white",
})

# Places in Jamaica used through the course, as [west, south, east, north].
PLACES = {
    "kingston":     [-76.86, 17.93, -76.80, 18.02],
    "black_river":  [-77.90, 17.96, -77.78, 18.08],
    "negril":       [-78.375, 18.25, -78.320, 18.36],
    "montego_bay":  [-77.97, 18.44, -77.88, 18.51],
    "new_hope":     [-78.20, 18.13, -78.08, 18.24],
    "st_elizabeth": [-77.75, 18.00, -77.65, 18.10],
    "portland":     [-76.45, 18.10, -76.32, 18.20],
    "jamaica":      [-78.45, 17.66, -76.15, 18.55],
}

def search_scenes(bbox, start, end, max_cloud=30, limit=50, sort_by="eo:cloud_cover",
                  min_cloud=None, descending=False):
    """Ask the archive which Sentinel-2 pictures exist over a box and a date range.

    Returns a list of STAC 'items'. Each item is a dictionary of metadata plus
    links to the actual image files. Nothing is downloaded yet.

    Set `min_cloud` when you deliberately want a cloudy scene, which is useful
    for testing that your cloud masking actually works.
    """
    cloud_filter = {"lt": max_cloud}
    if min_cloud is not None:
        cloud_filter["gt"] = min_cloud
    query = {
        "collections": ["sentinel-2-l2a"],
        "bbox": bbox,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "query": {"eo:cloud_cover": cloud_filter},
        "limit": limit,
        "sortby": [{"field": f"properties.{sort_by}",
                    "direction": "desc" if descending else "asc"}],
    }
    r = _stac_post(STAC_URL, query, timeout=60)
    return r.json()["features"]

def search_all(bbox, start, end, max_cloud=100, page_size=100, max_pages=20):
    """Every matching scene, not just the first page.

    The archive hands back at most 200 results per request and does not warn you
    that it stopped. This follows the 'next' link until the results run out.
    """
    body = {
        "collections": ["sentinel-2-l2a"], "bbox": bbox,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "query": {"eo:cloud_cover": {"lt": max_cloud}},
        "limit": page_size,
        "sortby": [{"field": "properties.datetime", "direction": "asc"}],
    }
    items, url, pages, matched = [], STAC_URL, 0, None
    while url and pages < max_pages:
        r = _stac_post(url, body, timeout=90)
        j = r.json()
        items += j.get("features", [])
        matched = j.get("context", {}).get("matched", matched)
        nxt = [l for l in j.get("links", []) if l.get("rel") == "next"]
        pages += 1
        if not nxt:
            break
        url = nxt[0]["href"]
        body = nxt[0].get("body", body)
    if matched and len(items) < matched:
        print(f"Warning: got {len(items)} of {matched}. Raise max_pages.")
    return items

def make_grid(bbox, metres=20):
    """Define a fixed grid of pixels over a box, in plain latitude and longitude.

    Every image we read gets warped onto this same grid. That is what lets us
    subtract a November picture from an October one pixel by pixel, even when
    the two came from different satellite tiles in different map projections.
    """
    lon0, lat0, lon1, lat1 = bbox
    shrink = math.cos(math.radians((lat0 + lat1) / 2))
    width  = int(round((lon1 - lon0) * 111320 * shrink / metres))
    height = int(round((lat1 - lat0) * 110540 / metres))
    transform = transform_from_bounds(lon0, lat0, lon1, lat1, width, height)
    return {"width": width, "height": height, "transform": transform,
            "metres": metres, "bbox": bbox,
            "pixel_hectares": (metres * metres) / 10000.0}

def read_band(item, band, grid, resampling=Resampling.bilinear):
    """Read one colour band of one scene onto our grid. Returns raw integers."""
    with rasterio.open(item["assets"][band]["href"]) as src:
        with WarpedVRT(src, crs="EPSG:4326", transform=grid["transform"],
                       width=grid["width"], height=grid["height"],
                       resampling=resampling) as vrt:
            return vrt.read(1)

def read_reflectance(item, band, grid):
    """Read a band and convert to reflectance (0 to 1). Divide by 10000."""
    return read_band(item, band, grid).astype("float32") / 10000.0

# Scene Classification Layer codes that mean 'this pixel is usable'.
# 4 vegetation, 5 bare soil, 6 water, 7 low-probability cloud, 11 snow/ice.
CLEAR_CODES = [4, 5, 6, 7, 11]

def clear_mask(item, grid):
    """True where the pixel is usable, False where it is cloud, shadow or edge."""
    scl = read_band(item, "scl", grid, Resampling.nearest)
    return np.isin(scl, CLEAR_CODES)

def check_coverage(item, grid):
    """How much of OUR area this scene actually covers, and how much is clear.

    The cloud percentage in the metadata describes the whole 110 km tile. It
    says nothing about your study area. Always check your own box.
    """
    scl = read_band(item, "scl", grid, Resampling.nearest)
    return {"covered": float((scl > 0).mean()),
            "clear": float(np.isin(scl, CLEAR_CODES).mean())}

def best_scene(items, grid, min_covered=0.95, min_clear=0.60, check_n=8):
    """Walk down the candidate list and return the first scene that is genuinely
    good over our box, not just good on paper."""
    for item in items[:check_n]:
        try:
            c = check_coverage(item, grid)
        except Exception:
            continue
        if c["covered"] >= min_covered and c["clear"] >= min_clear:
            item["_coverage"] = c
            return item
    return None

def composite(items, grid, bands, max_scenes=12, min_clear=0.10, verbose=True):
    """Stack several cloud-masked scenes and take the middle value per pixel.

    One picture of Jamaica almost always has cloud somewhere. Stack ten and take
    the median and the clouds disappear, because cloud is bright and rare while
    the ground underneath is consistent.
    """
    stacks = {b: [] for b in bands}
    used = []
    for item in items:
        if len(used) >= max_scenes:
            break
        try:
            clear = clear_mask(item, grid)
            if clear.mean() < min_clear:
                continue
            for b in bands:
                a = read_reflectance(item, b, grid)
                a[~clear] = np.nan
                a[a <= 0] = np.nan
                stacks[b].append(a)
            used.append(item["properties"]["datetime"][:10])
        except Exception:
            continue
    if not used:
        raise RuntimeError("No usable scenes found. Widen the dates or raise max_cloud.")
    if verbose:
        print(f"Composite built from {len(used)} scenes: {', '.join(sorted(used))}")
    out = {b: np.nanmedian(np.stack(v), axis=0) for b, v in stacks.items()}
    out["_dates"] = sorted(used)
    return out

def normalized_difference(a, b):
    """(a - b) / (a + b). The workhorse formula behind every index in this course."""
    return (a - b) / (a + b + 1e-10)

def stretch(rgb, low=2, high=98):
    """Rescale each colour channel so the picture is bright enough to look at."""
    out = np.zeros_like(rgb, dtype="float32")
    for i in range(rgb.shape[2]):
        band = rgb[:, :, i]
        p1, p2 = np.nanpercentile(band, [low, high])
        out[:, :, i] = np.clip((band - p1) / (p2 - p1 + 1e-9), 0, 1)
    return np.nan_to_num(out)

def show(image, title="", cmap=None, vmin=None, vmax=None, bar=False, size=(9, 8)):
    """Draw an array on screen with sensible defaults."""
    fig, ax = plt.subplots(figsize=size)
    im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    if bar:
        fig.colorbar(im, ax=ax, shrink=0.75)
    plt.tight_layout(); plt.show()

def area_hectares(mask, grid):
    """Convert a True/False mask into hectares on the ground."""
    return float(np.nansum(mask)) * grid["pixel_hectares"]

def label_frame(image_uint8, text):
    """Stamp a label bar onto one animation frame, so every frame says what it is."""
    img = Image.fromarray(image_uint8)
    draw = ImageDraw.Draw(img)
    bar = min(14 + 8 * len(text), img.width)
    draw.rectangle([0, 0, bar, 24], fill=(0, 0, 0))
    draw.text((7, 6), text, fill=(255, 255, 255))
    return np.array(img)

def save_gif(frames, path, ms=900):
    """Write labelled frames out as an animated GIF that loops forever."""
    import imageio.v2 as imageio
    imageio.mimsave(path, frames, duration=ms, loop=0)
    print(f"Saved {path}  ({os.path.getsize(path) / 1e6:.1f} MB, {len(frames)} frames)")

def show_gif(path):
    """Play a GIF inside the notebook."""
    try:
        from IPython.display import Image as _Gif, display
        display(_Gif(filename=path))
    except Exception:
        print("Open the file from the folder panel on the left to watch it.")

print("Toolkit loaded. Study areas available:", ", ".join(PLACES))

---

## Part 2. Why one before picture and one after picture can mislead

Grabbing one picture from before and one from after fails here three ways:
cloud trails the storm for weeks, no single scene covers Black River (the
Notebook 1 seam), and the answer would change with whichever two days you
happened to pick.

The fix is the median composite from Notebook 2. Stack five weeks of scenes
before landfall into one picture, four weeks after into another, and compare
periods instead of days.

In [ ]:
BLACK_RIVER = PLACES["black_river"]
br = make_grid(BLACK_RIVER, metres=20)

LANDFALL = "2025-10-28"
print(f"Study area: Black River, St Elizabeth")
print(f"Box: {BLACK_RIVER}")
print(f"Grid: {br['width']} by {br['height']} pixels at {br['metres']} m")
print(f"Ground covered: {br['width'] * br['height'] * br['pixel_hectares']:,.0f} hectares")
print(f"Landfall: {LANDFALL}")

### YOUR TURN 1

Set the two date windows. Melissa made landfall on **28 October 2025**.

Rules for choosing them:
- The before window must **end before** 28 October
- The after window must **start after** 28 October
- Neither window may contain the landfall date
- Each needs at least three weeks so there are enough scenes to stack

Use roughly five weeks before and four weeks after.

In [ ]:
# TODO: pick ONE of these three option sets, then copy its dates in.
# Two of them break a rule from the list above. The asserts will catch you.
#   A) before 2025-09-20 to 2025-10-27,  after 2025-10-30 to 2025-11-25
#   B) before 2025-10-01 to 2025-10-29,  after 2025-10-28 to 2025-11-10
#   C) before 2025-11-01 to 2025-11-20,  after 2025-09-01 to 2025-10-01
BEFORE_START, BEFORE_END = "____", "____"
AFTER_START,  AFTER_END  = "____", "____"

assert BEFORE_END < LANDFALL,  "The before window must end before landfall."
assert AFTER_START > LANDFALL, "The after window must start after landfall."
print(f"Before: {BEFORE_START} to {BEFORE_END}")
print(f"After : {AFTER_START} to {AFTER_END}")

*Hint: check each option against the rules. B lets the before window run past
landfall. C puts the after window before the storm ever arrived.*

---

## Part 3. Build a clean before picture and a clean after picture

This cell reads roughly seventy files. Give it two or three minutes. In the
live session, start it now and keep listening; it needs no babysitting.

In [ ]:
# 🚚 JUST RUN THIS CELL. It reads about seventy files; two to three minutes.
BANDS = ["green", "red", "nir"]
t0 = time.time()

print("BEFORE the storm")
before_scenes = search_scenes(BLACK_RIVER, BEFORE_START, BEFORE_END, max_cloud=70, limit=60)
before = composite(before_scenes, br, BANDS, max_scenes=12)

print("\nAFTER the storm")
after_scenes = search_scenes(BLACK_RIVER, AFTER_START, AFTER_END, max_cloud=70, limit=60)
after = composite(after_scenes, br, BANDS, max_scenes=12)

print(f"\nBuilt both in {time.time() - t0:.0f} seconds")

In [ ]:
ndvi_before = normalized_difference(before["nir"], before["red"])
ndvi_after  = normalized_difference(after["nir"],  after["red"])

# The bay and the river mouth are inside the box. Water has no vegetation to
# lose, so leaving it in would drag every average towards zero and invent damage
# where there was never anything growing.
water = normalized_difference(before["green"], before["nir"]) > 0.0
land = (~water) & np.isfinite(ndvi_before) & np.isfinite(ndvi_after)

land_ha = area_hectares(land, br)
print(f"Land in the study area: {land_ha:,.0f} hectares")
print(f"Coverage after stacking: {100 * land.mean() + 100 * water.mean():.1f}% of the box")

print(f"\nMean NDVI over land before: {np.nanmean(np.where(land, ndvi_before, np.nan)):.3f}")
print(f"Mean NDVI over land after:  {np.nanmean(np.where(land, ndvi_after,  np.nan)):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, arr, ttl in [(axes[0], ndvi_before, f"Before: {BEFORE_START} to {BEFORE_END}"),
                     (axes[1], ndvi_after,  f"After: {AFTER_START} to {AFTER_END}")]:
    im = ax.imshow(np.where(land, arr, np.nan), cmap="RdYlGn", vmin=0, vmax=0.9)
    ax.set_title(ttl, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.colorbar(im, ax=axes, shrink=0.7, label="NDVI")
fig.suptitle("Black River, St Elizabeth: vegetation either side of Hurricane Melissa",
             fontsize=13, fontweight="bold", y=0.97)
plt.show()

### The blink test: flick between before and after 🎞️

Two labelled frames, flicked forever. This is false colour, so living
vegetation burns red. Watch how much of the red goes out.

In [ ]:
fc_before = np.dstack([before["nir"], before["red"], before["green"]])
fc_after  = np.dstack([after["nir"],  after["red"],  after["green"]])

frames = [
    label_frame((stretch(np.nan_to_num(fc_before)) * 255).astype("uint8"),
                "BEFORE MELISSA  20 Sep to 27 Oct 2025"),
    label_frame((stretch(np.nan_to_num(fc_after)) * 255).astype("uint8"),
                "AFTER MELISSA   30 Oct to 25 Nov 2025"),
]

os.makedirs("outputs", exist_ok=True)
save_gif(frames, "outputs/melissa_blink.gif", ms=1000)
show_gif("outputs/melissa_blink.gif")

---

## Part 4. Subtract before from after to map the change

Subtract the two images and every pixel becomes a change value. Negative means
vegetation lost. The shared grid from Notebook 1 is what makes the subtraction
line up.

In [ ]:
# TODO: compute the change in NDVI. Negative should mean vegetation was lost.
# Options:  ndvi_after - ndvi_before  /  ndvi_before - ndvi_after  /  ndvi_after + ndvi_before
change = ____
change = np.where(land, change, np.nan)

mean_change = np.nanmean(change)
print(f"Average NDVI change over land: {mean_change:+.3f}")
assert mean_change < 0, "A hurricane should reduce vegetation. Check your subtraction order."

show(change, "NDVI change: red means vegetation lost",
     cmap="RdBu", vmin=-0.5, vmax=0.5, bar=True, size=(9, 8))

*Hint: after minus before. If a pixel went from 0.8 to 0.3, you want -0.5.*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
vals = change[np.isfinite(change)]
ax.hist(vals, bins=140, range=(-0.8, 0.4), color=CYAN, edgecolor="none")
ax.axvline(0, color="black", lw=1.4)
ax.axvline(-0.10, color=SAND, lw=1.6, ls="--", label="moderate loss (-0.10)")
ax.axvline(-0.20, color="#b3452e", lw=1.6, ls="--", label="severe loss (-0.20)")
ax.set_xlabel("Change in NDVI"); ax.set_ylabel("Number of pixels")
ax.set_title("How the whole study area shifted")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

print(f"Pixels that lost vegetation: {100 * (vals < 0).mean():.1f}%")
print(f"Pixels that gained:          {100 * (vals > 0).mean():.1f}%")

In an ordinary month that histogram is a narrow hump centred on zero. This one
has slid bodily left. That is what a Category 5 does.

---

## Part 5. Sort the damage into classes and count hectares

A relief convoy needs hectares, not a red map. Slice the change into severity
bands and count each. Part 6 tests whether the thresholds hold up.

In [ ]:
# TODO: complete the two missing thresholds
damage = {
    "Severe loss":    change < -0.30,
    "Heavy loss":    (change >= -0.30) & (change < ____),
    "Moderate loss": (change >= -0.20) & (change < -0.10),
    "Little change": (change >= -0.10) & (change <= 0.10),
    "Gained":         change > ____,
}

print(f"{'Severity':16s} {'Hectares':>10s} {'Share of land':>14s}")
print("-" * 44)
for name, mask in damage.items():
    ha = area_hectares(mask, br)
    print(f"{name:16s} {ha:10,.0f} {100 * ha / land_ha:13.1f}%")

affected = area_hectares(change < -0.10, br)
serious  = area_hectares(change < -0.20, br)
print(f"\nAffected at all (worse than -0.10): {affected:,.0f} ha "
      f"({100 * affected / land_ha:.0f}% of land)")
print(f"Seriously hit  (worse than -0.20): {serious:,.0f} ha "
      f"({100 * serious / land_ha:.0f}% of land)")

*Hint: the bands must join up with no gaps. Heavy loss runs from -0.30 up to
the moderate boundary. Gained starts where little change ends.*

In [ ]:
palette = {"Severe loss": "#7d1128", "Heavy loss": "#c0392b", "Moderate loss": "#e59866",
           "Little change": "#dfe6e9", "Gained": "#1e8449"}

painted = np.full(change.shape + (3,), np.nan)
for name, mask in damage.items():
    hexcode = palette[name].lstrip("#")
    painted[np.nan_to_num(mask.astype(float)).astype(bool)] = \
        [int(hexcode[j:j+2], 16) / 255 for j in (0, 2, 4)]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 7),
                             gridspec_kw={"width_ratios": [1, 0.85]})
a1.imshow(np.nan_to_num(painted, nan=1.0))
a1.set_title("Damage severity, Black River, October to November 2025")
a1.set_xticks([]); a1.set_yticks([]); a1.grid(False)

names = list(damage)
areas = [area_hectares(damage[n], br) for n in names]
a2.barh(names, areas, color=[palette[n] for n in names])
a2.invert_yaxis(); a2.set_xlabel("Hectares")
a2.set_title("Hectares in each class")
for i, v in enumerate(areas):
    a2.text(v, i, f" {v:,.0f}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

---

## Part 6. The control: prove it was the storm, not the season

October to November is Jamaica's second rainy season: vegetation normally
**greens up** across the exact window we measured. So how much of that -0.182
was Melissa, and how much was the calendar?

Run the identical analysis on 2023 and 2024, hurricane-free years. Whatever it
reports then is the **noise floor**: what the method says when nothing
happened. The most important three minutes in the notebook.

In [ ]:
def october_to_november(year, bbox, grid, verbose=False):
    """Run the whole before-and-after analysis for any year."""
    b = composite(search_scenes(bbox, f"{year}-09-20", f"{year}-10-27",
                                max_cloud=70, limit=60), grid,
                  ["green", "red", "nir"], max_scenes=12, verbose=verbose)
    a = composite(search_scenes(bbox, f"{year}-10-30", f"{year}-11-25",
                                max_cloud=70, limit=60), grid,
                  ["green", "red", "nir"], max_scenes=12, verbose=verbose)

    nb = normalized_difference(b["nir"], b["red"])
    na = normalized_difference(a["nir"], a["red"])
    wet = normalized_difference(b["green"], b["nir"]) > 0.0
    dry = (~wet) & np.isfinite(nb) & np.isfinite(na)

    # TODO: compute the change, keeping only land pixels.
    # Options:  na - nb  /  nb - na  /  na + nb
    ch = np.where(dry, ____, np.nan)

    return {"year": year,
            "before": float(np.nanmean(np.where(dry, nb, np.nan))),
            "after":  float(np.nanmean(np.where(dry, na, np.nan))),
            "change": float(np.nanmean(ch)),
            "ha_moderate": area_hectares(ch < -0.10, grid),
            "ha_severe":   area_hectares(ch < -0.20, grid),
            "land_ha":     area_hectares(dry, grid)}

results = []
for yr in [2023, 2024, 2025]:
    print(f"Running {yr}...")
    results.append(october_to_november(yr, BLACK_RIVER, br))

control = pd.DataFrame(results)
control["pct_moderate"] = 100 * control.ha_moderate / control.land_ha
control.round(3)

*Hint: same as Part 4. After minus before.*

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))

cols = [CYAN if y != 2025 else "#b3452e" for y in control.year]
a1.bar(control.year.astype(str), control.change, color=cols)
a1.axhline(0, color="black", lw=1)
a1.set_ylabel("Mean NDVI change, October to November")
a1.set_title("2023 and 2024 are the control years")
for i, v in enumerate(control.change):
    a1.text(i, v, f"{v:+.3f}", ha="center",
            va="bottom" if v > 0 else "top", fontsize=10)

a2.bar(control.year.astype(str), control.ha_moderate, color=cols)
a2.set_ylabel("Hectares worse than -0.10")
a2.set_title("Area flagged as damaged")
for i, v in enumerate(control.ha_moderate):
    a2.text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout(); plt.show()

normal = control[control.year != 2025]
storm  = control[control.year == 2025].iloc[0]
print(f"Normal years change by {normal.change.mean():+.3f} on average. Vegetation grows.")
print(f"2025 changed by {storm.change:+.3f}.")
print(f"\nNoise floor: the method flags {normal.ha_moderate.mean():,.0f} ha in a quiet year.")
print(f"In 2025 it flagged {storm.ha_moderate:,.0f} ha, "
      f"{storm.ha_moderate / normal.ha_moderate.mean():.0f} times more.")

In 2023 and 2024 vegetation **rises** through this window: the seasonal
green-up was pushing against the storm signal, and Melissa reversed it anyway.
The few hundred hectares flagged in quiet years is the false-positive rate,
which is what makes the 2025 number believable.

---

## 📍 Your spot: what did Melissa do to it?

Same spot as before. Two small composites, one honest number. About a minute.

In [ ]:
MY_LAT, MY_LON = ____, ____            # your spot again

spot = [MY_LON - 0.02, MY_LAT - 0.02, MY_LON + 0.02, MY_LAT + 0.02]
sg = make_grid(spot, metres=20)

sb = composite(search_scenes(spot, "2025-09-20", "2025-10-27", max_cloud=70, limit=40),
               sg, ["red", "nir"], max_scenes=8, verbose=False)
sa = composite(search_scenes(spot, "2025-10-30", "2025-11-25", max_cloud=70, limit=40),
               sg, ["red", "nir"], max_scenes=8, verbose=False)

d = float(np.nanmean(normalized_difference(sa["nir"], sa["red"])
                     - normalized_difference(sb["nir"], sb["red"])))
print(f"Your spot changed by {d:+.3f} NDVI across Melissa")
print("Quiet years sit near +0.03. Below -0.10 is real damage.")

---

## 🏁 Mission objective: severe loss in hectares around New Hope

New Hope, Westmoreland, is where Melissa's eye came ashore. The farms there fed
families and markets across the west of the island. This is the number a relief
agency needs first: not a picture of the damage, but its size.

**Find the value.** Hectares of land around New Hope that lost more than 0.20 of
their NDVI across the storm.

The composites take two to three minutes, so everyone starts together. Complete
the blank, run it, and call out your number. First correct answer wins.

In [ ]:
new_hope = PLACES["new_hope"]
ng = make_grid(new_hope, metres=20)

# 🚚 The next four lines just build the two composites. Run and wait.
b = composite(search_scenes(new_hope, "2025-09-20", "2025-10-27", max_cloud=70, limit=60),
              ng, ["green", "red", "nir"], max_scenes=12, verbose=False)
a = composite(search_scenes(new_hope, "2025-10-30", "2025-11-25", max_cloud=70, limit=60),
              ng, ["green", "red", "nir"], max_scenes=12, verbose=False)

nb_ = normalized_difference(b["nir"], b["red"])
na_ = normalized_difference(a["nir"], a["red"])
land_ = (normalized_difference(b["green"], b["nir"]) <= 0) & np.isfinite(nb_) & np.isfinite(na_)
ch_ = np.where(land_, na_ - nb_, np.nan)

# One blank: the severe-loss threshold. Options: -0.20 / 0.20 / -20
severe_ha = area_hectares(ch_ < ____, ng)
print(f"THE VALUE: {severe_ha:,.0f} hectares of severe loss")

*Stuck? Severe loss means NDVI fell a long way. A fall is a negative change, so
the threshold has to be a negative number, which rules out one option. And NDVI
only ever runs between -1 and 1, so a value of 20 is impossible, which rules out
another. That leaves one.*

---

## Mission 3 complete

- Composites make before-and-after comparisons repeatable instead of arbitrary
- A shared grid is what makes subtracting two dates mean anything
- Water must come out before you average vegetation
- Severity classes turn a picture into a number somebody can act on
- **A control period tells you what your method reports when nothing happened**
- Seasonal change can push against the signal you are chasing, or hide it
- MNDWI traces a coastline, and the tide moves it more than erosion does
- A trend smaller than its own uncertainty is not a finding

Next notebook: Kingston's temperature since 1981, machine learning that sorts
land cover on its own, and an honest attempt at predicting the future.

---

### Before you close this notebook

Save a copy to your own Drive (`File` then `Save a copy in Drive`). The next
notebook assumes you have this one working.

*Prepared by Adrian Dunkley, Climate Studies Group Mona, Faculty of Science and
Technology, University of the West Indies.*

*Satellite Data Analysis for Jamaica. Built with free, open data: Sentinel-2 from
the European Space Agency, hosted by Amazon; NASA POWER climate records. No API
keys, no fees, no permission needed.*